# Project Libraries Tour

A `Project` holds three libraries of reusable specs — **packages, grids,
simulations** — and a ledger of the **runs** made from them. This notebook is
that whole surface in one place: what is defined, what each one is, how to make
a variant, and which runs an edit invalidated.

Everything here runs on a tiny two-cell model in a temporary directory, so the
whole notebook executes in seconds and leaves nothing behind.

In [ ]:
import sys, tempfile
from pathlib import Path

# Make the in-repo `src/` importable when myflopy is not pip-installed.
src = Path.cwd().parents[2] / 'src'
if src.exists() and str(src) not in sys.path:
    sys.path.insert(0, str(src))

import myflopy as mf

root = Path(tempfile.mkdtemp()) / 'tour'
project = mf.Project(root, name='tour')
project

## 1 · Put the reusable pieces in the library

`add_package` / `add_grid` / `add_simulation` register under a **key you
choose**. The convention this project uses is `<type>/<variant>` — `npf/base`,
`rch/series` — which groups them on disk too (`specs/packages/npf/base.json`).

In [ ]:
project.add_package('dis/base', mf.dis(nlay=1, nrow=1, ncol=2,
                                       delr=100.0, delc=100.0, top=10.0, botm=0.0))
project.add_package('ic/steady', mf.ic(strt=8.0))
project.add_package('npf/base',  mf.npf(k=1.0))
project.add_package('chd/base',  mf.chd(stress_period_data={0: [((0, 0, 0), 9.0),
                                                                ((0, 0, 1), 7.0)]}))
project.add_package('oc/all',    mf.oc(head_filerecord='flow.hds',
                                       saverecord=[('HEAD', 'ALL')]))
project

## 2 · What is defined?

The libraries are **`dict` subclasses** — every dict operation still works, and
each one also answers the same three questions. Library keys contain `/`, so
they are not Python identifiers and can never be attribute-completed; they are
answered at runtime instead.

In [ ]:
project.packages.names()

In [ ]:
project.packages.summary()

In [ ]:
project.package('npf/base')          # same as project.packages['npf/base']

A miss names the near miss and everything defined, on the spelling you already
type:

In [ ]:
try:
    project.packages['npf/bass']
except KeyError as error:
    print(error.args[0])

## 3 · Build a simulation from references

`mf.ref(key)` points INTO the library rather than copying, so one package can be
shared by many simulations — and so the run manifest records which variant was
used.

In [ ]:
BASE = [mf.ref(k) for k in ('dis/base', 'ic/steady', 'npf/base', 'chd/base', 'oc/all')]

project.add_simulation(mf.SimulationSpec(
    'baseline',
    models=[mf.gwf('flow', packages=BASE)],
    packages=[mf.tdis(nper=1, perioddata=[(1.0, 1, 1.0)]), mf.ims(models=['flow'])],
))
project.simulations.summary()

## 4 · Make a variant

Two verbs, spelled apart on purpose: **`fork` leaves the original alone**;
**`edit` changes what an existing key means**.

Both are *built, not patched* — the new spec goes through the same constructor
`mf.npf(...)` does, so a typo raises here rather than at MF6 build time. And both
deep-copy by default, so editing a fork cannot write through into the original's
arrays.

In [ ]:
project.packages.fork('npf/base', 'npf/high_k', k=50.0, note='sensitivity test')
project.packages.summary()[['key', 'package', 'builder']]

In [ ]:
try:
    project.packages.fork('npf/base', 'npf/typo', kk_typo=1.0)
except TypeError as error:
    print(error)

Fork a **simulation** by swapping one library key for another. `swap` matches
the label an entry carries, which for `mf.ref` is its key.

In [ ]:
high_k = project.simulations.fork('baseline', 'high_k',
                                  swap={'npf/base': 'npf/high_k'})
print([getattr(e, 'key', None) or e.name for e in high_k.model('flow').packages])
print('derived_from:', high_k.derived_from)
print('lineage     :', [step['operation'] for step in high_k.lineage])

## 5 · Run them, and read what MF6 said

`status` is **myflopy's** verb — what the library did to the run. `outcome` is
**MF6's** verdict, read from `mfsim.lst`. They are two axes: a run written by one
session and executed by another sits at `written` forever while its listing says
`Normal termination`.

In [ ]:
for name in ('baseline', 'high_k'):
    project.run(name, project.simulations[name], overwrite=True)

project.refresh_runs().summary()[['key', 'status', 'outcome', 'stale', 'drift', 'elapsed']]

`run.reconcile()` adopts MF6's verdict into `status`/`success`. It refuses to
adopt one it cannot attribute — an interrupted run, or a listing that predates
this run's own preparation (`superseded`).

In [ ]:
run = project.reopen_run('baseline', load=False)
print('outcome  :', run.outcome)
print('succeeded:', run.outcome.succeeded)
print('packages :', run.package_keys)

## 6 · Which runs did an edit invalidate?

The manifest records package **keys**, not their contents — so editing
`npf/base` leaves every earlier run still naming it. Each run also records a
fingerprint of what those keys **meant** at the time.

Provenance is not content: stamping a note leaves everything `unchanged`.

In [ ]:
project.packages.edit('npf/base', note='reviewed')
project.refresh_runs().drift()

Changing a **value** does count, and names the runs it reaches:

In [ ]:
project.packages.edit('npf/base', k=3.0, note='post-calibration K')
project.refresh_runs().drift()

In [ ]:
project.runs.summary()[['key', 'status', 'outcome', 'drift']]

## 7 · Make it survive the session

`save()` writes what it can and returns a **`SaveReport`**, naming anything it
skipped. `pickled` is the number worth watching — a pickled entry is a cache,
not an archive.

In [ ]:
report = project.save()
report

In [ ]:
project.packages.persistence()

In [ ]:
later = mf.Project.load(root)          # a fresh session would start here
print(later)
print(later.packages.names())

# The manifest stores the rebuildable recipe, so a reopened run can be re-run or
# varied with no build script in sight.
reopened = later.reopen_run('baseline', load=False)
print('revived spec:', reopened.spec.name, reopened.spec.model_names)

## 8 · What is NOT a run

A PEST calibration builds into `<model workspace>.pest/`, holding the template,
the IES and prior masters, and every worker — each a full copy of the model
workspace. Listing them would add a dozen rows per calibration, so they are
excluded and **reported** instead.

In [ ]:
pest = project.runs_dir / 'baseline.pest'
for sub in ('cal', 'cal_ies_master',
            'cal_ies_master.workers/worker_0', 'cal_ies_master.workers/worker_1'):
    (pest / sub).mkdir(parents=True, exist_ok=True)
    (pest / sub / 'mfsim.nam').write_text('x')
(pest / 'cal' / 'myflopy_pest_metadata.json').write_text('{}')

runs = project.refresh_runs()
print(repr(runs))
runs.calibrations

Open them with `model.pest_runs` / `mf.find_pest_runs`, which return handles
that understand phi, ensembles and forecasts.

## 9 · Areas of interest

`project.aoi` is a fourth library: named **places to look at** — not model geometry.
`add_aoi` takes four numbers `(xmin, ymin, xmax, ymax)`, a run of points, any shapely
geometry, a GeoDataFrame or a vector file, and `crs=None` means the grid's frame. Every
model of the project shares it — `model.aoi` IS `project.aoi`, so `model.add_aoi(...)`
lands here — and on a DISV model a name then works wherever a place does:
`model.plot.map(extent='site')`, `select='site'`, `features=model.aoi`. It is saved
inline in the project spec, so an archive needs no file to restore it.

In [ ]:
project.add_aoi('west', (0.0, 0.0, 100.0, 100.0), note='the west cell', overwrite=True)
print(project.aoi['west'])
print(project.save())                      # aoi=1 -- written inline, no extra file
print(mf.Project.load(root).aoi.names())   # ...and back in a fresh session

---

## The whole grammar, in one table

| | |
|---|---|
| what is defined | `packages.names()`, `.summary()`, `repr(project)`, `project.summary()` |
| give me that one | `project.package(key)` / `.grid(key)` / `.simulation(name)`, or subscript |
| make a variant | `packages.fork(source, key, **overrides)`, `simulations.fork(source, name, swap=...)` |
| change one in place | `packages.edit(key, **overrides)` |
| what did I run | `project.runs.summary()`, `project.runs[key]`, `reopen_run(name)` |
| what did MF6 say | `run.outcome`, `run.reconcile()` |
| do its files still say what it recorded | `run.inputs`, the summary's `inputs` column |
| what did an edit break | `project.runs.drift()` |
| how would it persist | `packages.persistence()`, `project.save()` |
| name a place to look at | `project.add_aoi(name, place)` / `model.add_aoi(...)`, then `extent=name` |

Reference: [`docs/project_workflow.md`](../../../docs/project_workflow.md).